In [0]:
dbutils.fs.ls("/Volumes/healthcare_databricks/landing/landing_volume/neon_postgres/hospitals/2026-11-01_12-07-17/")

In [0]:
dbutils.widgets.text("source_system", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("source_file_format", "")

source_system = dbutils.widgets.get("source_system")
table_name = dbutils.widgets.get("table_name")
landing_timestamp = dbutils.widgets.get("landing_timestamp")
source_file_format = dbutils.widgets.get("source_file_format")

In [0]:
landing_path = f"/Volumes/healthcare_databricks/landing/landing_volume/{source_system}/{table_name}/{landing_timestamp}/"

# Check that the landing path (and its parent directories) exist before reading
def path_exists(path):
    try:
        dbutils.fs.ls(path)
        return True
    except Exception as e:
        if "FileNotFoundException" in str(e) or "PATH_NOT_FOUND" in str(e) or "does not exist" in str(e).lower():
            return False
        raise

if not path_exists(landing_path):
    msg = f"No new files received for {source_system}/{table_name} at {landing_timestamp}. Path does not exist: {landing_path}"
    print(msg)
    dbutils.notebook.exit(msg)

if source_file_format == "csv":
    df = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(landing_path)
else:
    df = spark.read.format("parquet").load(landing_path)

In [0]:
from pyspark.sql import functions as F

updated_df = df.withColumn("landing_timestamp", F.lit(landing_timestamp)) \
    .withColumn("insert_timestamp", F.current_timestamp())

In [0]:
updated_df.write.mode("append").saveAsTable(f"healthcare_databricks.bronze.{table_name}")